# Anatomy of an App -- From Notebook to Dashboard

01-05 taught you Python, Plotly, pandas, pulling data from NOMAD, and the building blocks of a Voila UI. This notebook is the missing link: how those pieces become a real app that shows up as a card on the App Dashboard.

The worked example throughout is **Design of Experiments** (`apps/DesignOfExperiments/`) -- a peroTF app that is split into separate Python modules, one job each, and that needs no NOMAD login, so everything here works offline. Every code cell below opens the *actual* files on disk, so what you see is always current, not a snapshot that can drift out of date.

## 0. Running code cells

- Run a cell with **Shift + Enter**
- This notebook needs to run from inside the repo (`Learning/` sitting next to `apps/`) -- if a cell can't find a file, check your working directory.


In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent
APP_DIR = REPO_ROOT / "apps" / "DesignOfExperiments"
assert APP_DIR.is_dir(), f"Can't find {APP_DIR} -- run this notebook from Learning/"

print("Repo root:", REPO_ROOT)

## 1. One job per file, and why

The larger apps in `apps/<AppName>/` are split into several Python files, each with one job. Design of Experiments has six:

| File | Job | Imports `ipywidgets`? |
|---|---|---|
| `data_manager.py` | the experiment variables (`Variable`, `VariableType`), validation, import/export | **No** |
| `sampling_algorithms.py` | the sampling methods (Latin hypercube, Sobol, Halton, grid, ...) and their quality metrics | **No** |
| `plot_manager.py` | build the Plotly figures | **No** |
| `utils.py` | constants and validation/helper functions shared by the others | **No** |
| `gui_components.py` | the widgets, forms and buttons | Yes |
| `app.py` | `DoEApplication`: builds the tabs and wires the other modules together | Yes |

The point of the split: everything without widgets can be imported and unit tested with plain `pytest` -- no notebook, no browser involved. Only `gui_components.py` and `app.py` need the interactive environment. Some apps (`Wetting_envelope`, the `Hansen_green_calculator` notebooks, ...) still keep all their code inside one notebook; that works, but nothing in it can be tested or reused without running the notebook.

Here's each file's own docstring, together with a check of whether it really imports `ipywidgets`:

In [ ]:
MODULES = [
    "data_manager.py",
    "sampling_algorithms.py",
    "plot_manager.py",
    "utils.py",
    "gui_components.py",
    "app.py",
]

for name in MODULES:
    text = (APP_DIR / name).read_text(encoding="utf-8")
    docstring = text.split('"""')[1].strip() if '"""' in text[:20] else "(no module docstring)"
    uses_widgets = "import ipywidgets" in text
    print(f"--- {name} (imports ipywidgets: {uses_widgets}) ---")
    print(docstring)
    print()

## 2. The notebook is just the start button

Every app's `<app>.ipynb` starts with the same two code cells: cell 0 runs the repo's `bootstrap.py`, cell 1 logs notebook usage. The next code cell imports the app's main class and displays it; markdown cells for a title or a short guide are fine in between. Here's Design of Experiments' actual notebook, cell by cell:

In [ ]:
import json

nb = json.loads((APP_DIR / "DoE.ipynb").read_text(encoding="utf-8"))
print(f"{len(nb['cells'])} cells\n")
for i, cell in enumerate(nb["cells"]):
    print(f"--- cell {i} ({cell['cell_type']}) ---")
    print("".join(cell["source"]))
    print()

That's the whole notebook -- all the real logic lives in the `.py` files, not here.

Cell 0 is the one cell every app shares: `runpy.run_path("../../bootstrap.py")` runs `bootstrap.py` from the repo root, which installs `perotf_utils` from `shared/` *and* puts it on `sys.path` for the running kernel (a plain `pip install` alone would only become importable after a kernel restart), then installs the dependencies listed in the app's own `pyproject.toml`. That is why no app file ever touches `sys.path` itself. The `_ =` matters too: `run_path` returns bootstrap's whole namespace, and without the assignment the notebook would display that dictionary at the top of the app.

## 3. Getting it onto the dashboard

The App Dashboard doesn't discover apps automatically -- each card is one `AppEntry` in `apps/App_dashboard/data_manager.py`'s `CATEGORIES` dict. Here's the real entry for Design of Experiments:

In [ ]:
import re

dashboard_src = (REPO_ROOT / "apps" / "App_dashboard" / "data_manager.py").read_text(
    encoding="utf-8"
)
match = re.search(r'AppEntry\(\s*"DesignOfExperiments".*?\n\s*\),', dashboard_src, re.DOTALL)
print(match.group())

Field by field:

- `folder` -- the app's directory name under `apps/`
- `notebook` -- which `.ipynb` in that folder to render
- `name` / `description` / `icon` -- what shows on the card (icon is any [Font Awesome](https://fontawesome.com/search?o=r&m=free) class name)
- `experimental=True` (optional) -- adds an "experimental" badge for apps that are not finished yet

Add an `AppEntry` like this one to the right category and your app has a card on the dashboard -- that's the entire registration step. One folder can have several cards (`Hansen_green_calculator` registers one per notebook), and a test in `tests/App_dashboard/` fails if an app folder has no card or a card points at a notebook that doesn't exist.

## 4. Packaging: `pyproject.toml`

Each app has its own `pyproject.toml`. Its `dependencies` list is not decoration: `bootstrap.py` installs the app's folder with it when the app starts on the Oasis, so every third-party package the app imports has to be listed (a missing one shows up there as a `ModuleNotFoundError`). `perotf-utils` is listed as a plain name, without a path -- `bootstrap.py` has already installed it from `shared/` by then. The two `[tool.hatch...]` sections let the build tool package a flat folder of modules:

In [ ]:
print((APP_DIR / "pyproject.toml").read_text(encoding="utf-8"))

## 5. Tests

Modules without widgets can be imported and tested with plain `pytest` -- no notebook involved. Tests live in `tests/<AppName>/` (never inside `apps/`). Since every app has its own `data_manager.py`, `app.py`, ..., importing two apps' modules by their bare names in one test run would mix them up. The root `tests/conftest.py` therefore provides an `app_loader` fixture that imports one app's modules and removes their bare names again afterwards. Each app's `conftest.py` just asks it for its modules, and the tests receive them through a fixture instead of `import data_manager`:

In [ ]:
TEST_DIR = REPO_ROOT / "tests" / "DesignOfExperiments"

for path in [TEST_DIR / "conftest.py", *sorted(TEST_DIR.glob("test_*.py"))]:
    print(f"--- {path.relative_to(REPO_ROOT).as_posix()} ---")
    print(path.read_text(encoding="utf-8"))

⚠️ **One repo-specific gotcha, worth knowing before it costs you an hour:** the repo root's own `secrets.py` (the local-testing token file) shadows Python's *stdlib* `secrets` module whenever the repo root ends up first on `sys.path` -- which breaks `numpy`/`pandas`/`plotly` imports with a confusing `ImportError: cannot import name 'randbits' from 'secrets'`. `tests/conftest.py` strips the repo root from `sys.path` specifically to prevent this. If that error ever resurfaces, check that file still exists before debugging anything else.


## 6. Landing the change: issues, PRs, versions

Condensed from `CONTRIBUTING.md` at the repo root (read it in full before your first PR):

1. **Open an issue first** on [`NOMADe-Joshua/Voila-Apps-V2`](https://github.com/NOMADe-Joshua/Voila-Apps-V2/issues) -- bug fixes and features both start here. (Typo fixes and tooling-only changes are exempt.)
2. **Open a PR that references it**, e.g. `Fixes #123` in the description.
3. **Bump that app's version** in its own `pyproject.toml`, SemVer: patch for a fix, minor for a backward-compatible feature, major for a breaking change. Bump only the app you changed.
4. Releases are cut with `gh release create <tag> --generate-notes`, which builds notes from merged PR titles -- so a descriptive PR title *is* your changelog entry.


## Exercises (recommended)

1. Compare `apps/DesignOfExperiments/` with `apps/Wetting_envelope/` (all code in one notebook) and with `apps/XRD_PF/` or `apps/UVVis_Analyzer/` (split into modules like Design of Experiments). Which parts of each could you test without opening a notebook?
2. Sketch (in words, no need to write real code) what `data_manager.py` / `plot_manager.py` / `gui_components.py` would each contain for a tiny app that plots `y = x**n` for a user-chosen `n`.
3. Write the `AppEntry(...)` you'd add to `CATEGORIES` for that fictional app.
4. Open `tests/DesignOfExperiments/test_designofexperiments.py`, then add one test of your own for a widget-free function, e.g. `format_percentage` in `apps/DesignOfExperiments/utils.py`, getting the module from the `mods` fixture.